# Commodity Currency Portfolios
**Block 4 - Commodity-beta sorted currency portfolios**

Sorts 12 commodity currencies into equal-weighted tercile portfolios by 24-month rolling commodity beta.

- **CUR1**: low commodity beta (least sensitive to commodity prices)
- **CUR2**: medium commodity beta
- **CUR3**: high commodity beta (most sensitive to commodity prices)
- **rCURR**: CUR3 − CUR1 (long-short factor)

**Dependency:** Run `commodity_futures_portfolios.ipynb` through its Save Outputs section first, to generate `Output/commodity_returns_weekly.csv` (referencing a cell number here is fragile -- that notebook's cells have been renumbered multiple times; the output filename is stable, the index isn't).


In [ ]:
# Configuration
import pandas as pd
import numpy as np
import os

CURRENCY_FILE = 'Input/commodity_currencies_data.xlsx'
COMM_RET_FILE = 'Output/commodity_returns_weekly.csv'  # written by commodity_futures_portfolios.ipynb

# Currencies requiring inversion: quote is FCY per USD → convert to USD per FCY
INVERT = {
    'CNDOLL$', 'NORKRO$', 'COMRAN$', 'BRACRU$',
    'CHILPE$', 'CISRUB$', 'MEXPES$', 'PERUSO$',
    'MALADL$', 'INDORU$',
}
# AUD (AUSTDO$) and NZD (NZDOLL$) are already quoted as USD per FCY - no inversion needed

CURRENCY_NAMES = {
    'AUSTDO$': 'AUD', 'CNDOLL$': 'CAD', 'NZDOLL$': 'NZD',
    'NORKRO$': 'NOK', 'COMRAN$': 'ZAR', 'BRACRU$': 'BRL',
    'CHILPE$': 'CLP', 'CISRUB$': 'RUB', 'MEXPES$': 'MXN',
    'PERUSO$': 'PEN', 'MALADL$': 'MYR', 'INDORU$': 'IDR',
}

BETA_WINDOW  = 104    # weeks -> 24 months rolling OLS window
MIN_OBS      = 24     # minimum valid weekly obs required per regression
STUDY_START  = '1995-01-04'
STUDY_END    = '2023-12-27'

inverted_names = [CURRENCY_NAMES[c] for c in INVERT if c in CURRENCY_NAMES]
print(f"Currencies      : {len(CURRENCY_NAMES)}")
print(f"Inverted (→ USD/FCY): {inverted_names}")
print(f"Beta window     : {BETA_WINDOW} weeks (~24 months)")


Currencies      : 12
Inverted (→ USD/FCY): ['RUB', 'PEN', 'NOK', 'CLP', 'MYR', 'MXN', 'IDR', 'BRL', 'CAD', 'ZAR']
Beta window     : 104 weeks (~24 months)


In [4]:
# Load currency data
meta     = pd.read_excel(CURRENCY_FILE, header=None, nrows=6, dtype=str)
code_row = meta.iloc[4, :]
codes    = [v for v in code_row.dropna().tolist() if v != 'Code']

raw = pd.read_excel(CURRENCY_FILE, header=None, skiprows=6,
                    usecols=range(len(codes) + 1), index_col=0)
raw.index      = pd.to_datetime(raw.index, errors='coerce')
raw.index.name = 'date'
raw.columns    = codes
raw            = raw.sort_index().apply(pd.to_numeric, errors='coerce')

# Standardise all quotes to USD per FCY
fx = raw.copy()
for code in codes:
    if code in INVERT:
        fx[code] = 1.0 / fx[code]

fx = fx.rename(columns=CURRENCY_NAMES)

print(f"FX data  : {fx.shape[0]} daily obs x {fx.shape[1]} currencies")
print(f"Period   : {fx.index.min().date()} - {fx.index.max().date()}")
print(f"\nFirst valid date per currency:")
for c in fx.columns:
    fv = fx[c].first_valid_index()
    print(f"  {c}: {fv.date() if fv is not None else 'N/A'}")

FX data  : 8088 daily obs × 12 currencies
Period   : 1993-12-31 – 2024-12-31

First valid date per currency:
  AUD: 1993-12-31
  CAD: 1993-12-31
  NZD: 1993-12-31
  NOK: 1993-12-31
  ZAR: 1993-12-31
  BRL: 1994-06-30
  CLP: 1993-12-31
  RUB: 1996-03-06
  MXN: 1993-12-31
  PEN: 1993-12-31
  MYR: 1993-12-31
  IDR: 1993-12-31


In [5]:
# Weekly FX returns (Wednesday close, log-compounding)
log_daily  = np.log(fx).diff()
week_id    = fx.index.to_period('W-WED')
log_wkly   = log_daily.groupby(week_id).sum(min_count=1)
ret_weekly = np.expm1(log_wkly)
# .normalize() strips the sub-day time component (to_timestamp(how='end')
# produces 23:59:59.999999, not midnight) so the index aligns with the
# commodity_returns_weekly.csv index (resample('W-WED') → midnight timestamps).
ret_weekly.index      = ret_weekly.index.to_timestamp(how='end').normalize()
ret_weekly.index.name = 'date'

print(f"Weekly FX returns: {ret_weekly.shape[0]} weeks x {ret_weekly.shape[1]} currencies")
print(f"Period           : {ret_weekly.index.min().date()} - {ret_weekly.index.max().date()}")
print(f"\nCoverage (fraction non-missing):")
print(ret_weekly.notna().mean().round(2).to_string())


Weekly FX returns: 1618 weeks x 12 currencies
Period           : 1994-01-05 - 2025-01-01

Coverage (fraction non-missing):
AUD    1.00
CAD    1.00
NZD    1.00
NOK    1.00
ZAR    1.00
BRL    0.98
CLP    1.00
RUB    0.93
MXN    1.00
PEN    1.00
MYR    1.00
IDR    1.00


In [6]:
# Build commodity index
# Equal-weighted average of individual commodity futures weekly returns.
# Loaded from commodity_returns_weekly.csv (commodity_futures_portfolios.ipynb cell 8).

try:
    comm_ret = pd.read_csv(COMM_RET_FILE, index_col=0, parse_dates=True)
    print(f"Loaded commodity returns: {comm_ret.shape}")
    print(f"Period: {comm_ret.index.min().date()} - {comm_ret.index.max().date()}")

except FileNotFoundError:
    print(f"File not found: {COMM_RET_FILE}")
    print("Run commodity_futures_portfolios.ipynb cell 8 first.")
    raise

# Equal-weighted commodity index (all available F1 returns)
comm_idx = comm_ret.mean(axis=1).rename('comm_idx')
comm_idx = comm_idx.reindex(ret_weekly.index)

print(f"\nCommodity index: {comm_idx.notna().sum()} non-missing weekly obs")
print(f"Period         : {comm_idx.first_valid_index().date()} - {comm_idx.last_valid_index().date()}")
print(f"Mean ann. return: {comm_idx.mean() * 52 * 100:.2f}%")

Loaded commodity returns: (1774, 27)
Period: 1990-01-03 - 2023-12-27

Commodity index: 1565 non-missing weekly obs
Period         : 1994-01-05 - 2023-12-27
Mean ann. return: 0.68%


In [7]:
# 24-month rolling commodity betas
# At each month-end t, regress prior 104 weeks of weekly currency returns
# on the equal-weighted commodity index return.
# β = commodity sensitivity used to sort currencies into tercile portfolios.

aligned = ret_weekly.join(comm_idx, how='left')

# Month-end rebalancing dates (need ≥ BETA_WINDOW weeks of history)
month_ends = pd.date_range(
    start = aligned.index.min() + pd.DateOffset(weeks=BETA_WINDOW),
    end   = aligned.index.max(),
    freq  = 'ME'
)

beta_records = []

for me in month_ends:
    # Last BETA_WINDOW Wednesdays up to and including month-end
    wednesdays = aligned.index[aligned.index <= me]
    if len(wednesdays) < BETA_WINDOW:
        continue
    window = aligned.loc[wednesdays[-BETA_WINDOW:]]

    x_raw    = window['comm_idx'].values
    valid_x  = ~np.isnan(x_raw)
    x_design = np.column_stack([np.ones(valid_x.sum()), x_raw[valid_x]])

    row = {'date': me}
    for ccy in ret_weekly.columns:
        y_raw   = window[ccy].values[valid_x]
        valid_y = ~np.isnan(y_raw)
        if valid_y.sum() < MIN_OBS:
            row[ccy] = np.nan
            continue
        xv, yv = x_design[valid_y], y_raw[valid_y]
        try:
            beta = np.linalg.lstsq(xv, yv, rcond=None)[0][1]
        except np.linalg.LinAlgError:
            beta = np.nan
        row[ccy] = beta

    beta_records.append(row)

beta_df = pd.DataFrame(beta_records).set_index('date')

print(f"Rolling betas: {beta_df.shape[0]} months x {beta_df.shape[1]} currencies")
print(f"Period       : {beta_df.index.min().date()} - {beta_df.index.max().date()}")
print(f"\nMean commodity beta per currency (sorted):")
print(beta_df.mean().sort_values(ascending=False).round(3).to_string())

Rolling betas: 348 months x 12 currencies
Period       : 1996-01-31 - 2024-12-31

Mean commodity beta per currency (sorted):
BRL    0.321
AUD    0.321
NOK    0.316
ZAR    0.302
NZD    0.262
CAD    0.212
CLP    0.199
MXN    0.191
RUB    0.184
MYR    0.137
IDR    0.105
PEN    0.061


In [8]:
# Monthly sort and weekly portfolio returns

def rank_terciles(row, n=3):
    """Assign equal-frequency tercile labels 1/2/3 across currencies for one month."""
    valid = row.dropna()
    if len(valid) < n:
        return pd.Series(np.nan, index=row.index)
    try:
        labels = pd.qcut(valid.rank(method='first'), q=n, labels=False) + 1
    except ValueError:
        labels = pd.cut(valid.rank(method='average'), bins=n, labels=False) + 1
    return labels.astype(float).reindex(row.index)

tercile_df = beta_df.apply(rank_terciles, axis=1)

# Long-format tercile assignments (formed at month-end t, applied to month t+1)
sort_long = (tercile_df
             .stack()
             .reset_index()
             .rename(columns={0: 'tercile', 'level_1': 'currency'}))
sort_long['year']  = sort_long['date'].dt.year
sort_long['month'] = sort_long['date'].dt.month

# Long-format weekly returns labelled by prior month (formation month)
ret_long = (ret_weekly
            .stack()
            .reset_index()
            .rename(columns={0: 'ret', 'level_1': 'currency'}))
ret_long['year']  = (ret_long['date'].dt.to_period('M') - 1).dt.year
ret_long['month'] = (ret_long['date'].dt.to_period('M') - 1).dt.month

# Merge returns with tercile assignments
merged = ret_long.merge(
    sort_long[['year', 'month', 'currency', 'tercile']],
    on=['year', 'month', 'currency'],
    how='inner'
)

# Equal-weighted portfolio returns per tercile
port_ret = (merged
            .groupby(['date', 'tercile'])['ret']
            .mean()
            .unstack('tercile'))
port_ret.columns = [f'CUR{int(c)}' for c in port_ret.columns]
port_ret = port_ret.sort_index().loc[STUDY_START:STUDY_END]

print(f"Currency portfolios: {port_ret.shape}")
print(f"Period             : {port_ret.index.min().date()} - {port_ret.index.max().date()}")
print(f"\nCoverage (fraction non-missing):")
print(port_ret.notna().mean().round(2).to_string())
print(f"\nAvg currencies per tercile per month:")
print(merged.groupby(['date','tercile'])['currency'].count()
           .groupby('tercile').mean().round(1).to_string())

Currency portfolios: (1456, 3)
Period             : 1996-02-07 - 2023-12-27

Coverage (fraction non-missing):
CUR1    1.0
CUR2    1.0
CUR3    1.0

Avg currencies per tercile per month:
tercile
1.0    4.0
2.0    4.0
3.0    4.0


In [ ]:
# RF subtraction
import pandas_datareader.data as web

rf_raw = web.DataReader('F-F_Research_Data_Factors_weekly',
                        'famafrench', start='1990-01-01')[0]
rf_wk  = rf_raw['RF'] / 100

# Fix index - may be PeriodIndex or DatetimeIndex depending on version
if hasattr(rf_wk.index, 'to_timestamp'):
    rf_wk.index = pd.to_datetime(rf_wk.index.to_timestamp(how='end'))
else:
    rf_wk.index = pd.to_datetime(rf_wk.index)
rf_wk.index = rf_wk.index.normalize()
rf_wk.index.name = 'date'

# RF is Friday-end; port_ret is Wednesday-end.
# Reindex to a daily grid then forward-fill, then select Wednesdays.
daily_idx = pd.date_range(rf_wk.index.min(), port_ret.index.max(), freq='D')
rf_daily  = rf_wk.reindex(daily_idx).ffill()
rf_al     = rf_daily.reindex(port_ret.index)

print(f"RF aligned: {rf_al.notna().sum()} non-NaN out of {len(rf_al)}")

excess = port_ret.subtract(rf_al, axis=0)
excess['rCURR'] = excess['CUR3'] - excess['CUR1']

print("\nCurrency portfolio excess returns:")
print(f"  Shape  : {excess.shape}")
print(f"\nAnnualised mean excess return (%):")
print((excess.mean() * 52 * 100).round(2).to_string())
print(f"\nAnnualised volatility (%):")
print((excess.std() * np.sqrt(52) * 100).round(2).to_string())

C:\Users\fdxja\AppData\Local\Temp\ipykernel_28308\1090638023.py:4: FutureWarning: The argument 'date_parser' is deprecated and will be removed in a future version. Please use 'date_format' instead, or read your data in as 'object' dtype and then call 'to_datetime'.
  rf_raw = web.DataReader('F-F_Research_Data_Factors_weekly',


RF aligned: 1456 non-NaN out of 1456

Currency portfolio excess returns:
  Shape  : (1456, 4)

Annualised mean excess return (%):
CUR1    -6.05
CUR2    -4.78
CUR3    -3.11
rCURR    2.94

Annualised volatility (%):
CUR1      6.56
CUR2      8.42
CUR3     11.99
rCURR    10.47


In [11]:
# Save
excess.to_csv('Output/commodity_currency_weekly.csv')

print(f"Saved: commodity_currency_weekly.csv - {excess.shape}")
print(f"  CUR1  : low commodity beta")
print(f"  CUR2  : medium commodity beta")
print(f"  CUR3  : high commodity beta")
print(f"  rCURR : CUR3 − CUR1 (long-short factor)")

Saved: commodity_currency_weekly.csv - (1456, 4)
  CUR1  : low commodity beta
  CUR2  : medium commodity beta
  CUR3  : high commodity beta
  rCURR : CUR3 − CUR1 (long-short factor)
